# Web IQ: MCP 서버 직접 호출

## 목표와 사전 준비

포털 밖의 Python에서 Web IQ MCP 서버에 연결해 도구 목록을 조회하고 검색 도구를 직접 호출합니다. REST와 동일하게 Web, Images, Videos에 같은 검색어를 전달하며 `YouTube`를 자동으로 덧붙이지 않습니다. **고객 실습에 사용이 허용된 진행자 제공 키로 참가자가 직접 실행**합니다. 키가 없으면 [설명 자료](README.md)의 기존 캡처를 분석하며 직접 호출을 완료했다고 기록하지 않습니다.

- Python 3.11 이상, Jupyter, `web`, `images`, `videos` 권한이 있는 Web IQ 키가 필요합니다.
- [REST 노트북](01_rest_api.ipynb)에서 사용한 키를 `WEBIQ_API_KEY` 환경 변수 또는 숨김 입력으로 전달합니다.
- 실행 위치는 이 노트북입니다. 위에서 아래로 실행합니다. `await`는 Jupyter에서 직접 사용하며 `asyncio.run()`으로 감싸지 않습니다.
- 공식 서버: `https://api.microsoft.ai/v3/mcp`. 인증 헤더는 `x-apikey`입니다.
- LLM이 도구를 자동 선택하는 예제가 아닙니다. 클라이언트가 명시적으로 MCP 도구를 호출해 REST와 같은 결과 필드를 확인합니다.
- [공식 MCP 문서](https://webiq.microsoft.ai/documentation/mcp/)에 따르면 도구 목록은 계정 권한에 따라 달라집니다. 없는 도구는 권한을 확인하고 임의로 우회하지 않습니다.

흐름: 연결 → `initialize()` → `list_tools()` → `call_tool()` → 연결 종료. 검색 호출은 총 6회이며 할당량·비용은 계정 정책을 따릅니다.

In [1]:
%pip install "mcp==1.30.0" "httpx>=0.28,<1"

Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Users/changjuahn/Repo/MicrosoftIQSeries-KR/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import json
import os
from datetime import datetime, timezone

import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

api_key = (os.environ.get("WEBIQ_API_KEY") or getpass.getpass("Web IQ API key: ")).strip()
if not api_key:
    raise ValueError("Web IQ API 키가 필요합니다.")
print("API 키 준비 완료. 키 값은 출력하지 않습니다.")
print("실행 시각 (UTC):", datetime.now(timezone.utc).isoformat())

API 키 준비 완료. 키 값은 출력하지 않습니다.
실행 시각 (UTC): 2026-09-30T01:34:04.476575+00:00


In [ ]:
mcp_results = {}
result_fields = {"web": "webResults", "images": "imageResults", "videos": "videoResults"}

async with httpx.AsyncClient(headers={"x-apikey": api_key}, timeout=60.0) as http_client:
    async with streamable_http_client(
        "https://api.microsoft.ai/v3/mcp", http_client=http_client
    ) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            initialized = await session.initialize()
            print("MCP server:", initialized.serverInfo.name)
            print("Protocol:", initialized.protocolVersion)
            tools = await session.list_tools()
            tool_names = {tool.name for tool in tools.tools}
            print("tools/list:", sorted(tool_names))
            missing = set(result_fields) - tool_names
            if missing:
                raise RuntimeError(f"필수 검색 도구 권한이 없습니다: {sorted(missing)}")

            for topic in ("Microsoft Foundry", "Microsoft Copilot"):
                mcp_results[topic] = {}
                for tool_name, result_field in result_fields.items():
                    arguments = {"query": topic, "maxResults": 3, "language": "en", "region": "US"}
                    if tool_name == "web":
                        arguments.update(contentFormat="passage", maxLength=800)

                    result = await session.call_tool(tool_name, arguments=arguments)
                    print(f"\ntools/call: {tool_name} | {topic} | isError={result.isError}")
                    if result.isError:
                        raise RuntimeError(f"MCP {tool_name} 도구가 오류를 반환했습니다. 권한·매개변수·할당량을 확인하세요.")
                    payload = result.structuredContent
                    if payload is None:
                        text_blocks = [block.text for block in result.content if block.type == "text"]
                        if not text_blocks:
                            raise RuntimeError("MCP 구조화 결과와 JSON 텍스트가 모두 없습니다.")
                        payload = json.loads("\n".join(text_blocks))
                    items = payload.get(result_field, [])
                    if not items:
                        raise RuntimeError(f"{topic} / {tool_name}: 반환 결과가 없습니다.")
                    mcp_results[topic][tool_name] = payload
                    print(f"{result_field}: {len(items)}건 | traceId: {payload.get('traceId', '(없음)')}")
                    for item in items:
                        print("제목:", item.get("title", "(없음)"))
                        print("URL:", item.get("url", "(없음)"))
                        print("내용:", (item.get("content") or item.get("caption") or item.get("description") or "")[:220])
                        if tool_name == "images":
                            print("원문 페이지:", item.get("hostPageUrl", "(없음)"))
                        if tool_name == "videos":
                            print("게시자:", item.get("publishedBy", "(없음)"))
print("MCP 세션과 HTTP 연결을 종료했습니다.")

## 결과 확인과 정리

- 연결 성공 시 MCP 서버 이름, 협상한 프로토콜 버전과 `tools/list` 결과가 출력됩니다.
- 여섯 검색에서 `isError=False`와 비어 있지 않은 결과 목록을 확인합니다. REST와 동일한 검색어·매개변수를 사용해도 검색 시점에 따라 순위와 결과는 달라질 수 있습니다.
- `structuredContent`가 없으면 공식 문서의 JSON 텍스트 대체 결과를 읽습니다. 응답에서 생략된 목록은 빈 목록으로 취급하며 이 데모에서는 검증 오류로 표시합니다.
- `Videos`는 YouTube 전용 서비스가 아닙니다. 다른 Vertical과 같은 검색어를 사용하며 실제 결과 URL과 게시자를 확인해야 합니다.
- 401/403은 키·서비스 권한, 429는 할당량을 확인합니다. `isError=True`는 HTTP 연결 성공과 별개인 도구 실행 실패입니다.

`async with`가 끝나면 세션과 HTTP 연결이 닫힙니다. 두 노트북 실습이 끝나면 커널을 재시작하고 환경 변수를 제거합니다. Profile Management에서는 이번에 만든 데모 전용 키만 폐기하고 기존 기본 키는 유지하세요.